In [ ]:
import numpy as np
import emcee
import corner  # for visualization

# Your Lorenz 63 simulator
def lorenz63_trajectory(sigma, rho, beta, x0, T, dt):
    """Simulate Lorenz 63"""
    n_steps = int(T / dt)
    trajectory = np.zeros((n_steps, 3))
    trajectory[0] = x0
    
    for i in range(1, n_steps):
        x, y, z = trajectory[i-1]
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z
        trajectory[i] = trajectory[i-1] + dt * np.array([dx, dy, dz])
    
    return trajectory

# Bayesian inference setup
def log_prior(theta):
    sigma, rho, beta = theta
    # Uniform priors (adjust to match your training distribution)
    if 5 < sigma < 15 and 20 < rho < 35 and 1 < beta < 4:
        return 0.0
    return -np.inf

def log_likelihood(theta, observed_traj, x0, T, dt, noise_std=0.1):
    sigma, rho, beta = theta
    
    # Simulate
    sim_traj = lorenz63_trajectory(sigma, rho, beta, x0, T, dt)
    
    # Assume Gaussian observation noise
    residuals = observed_traj - sim_traj
    log_like = -0.5 * np.sum(residuals**2 / noise_std**2)
    
    return log_like

def log_probability(theta, observed_traj, x0, T, dt):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, observed_traj, x0, T, dt)

# Generate test data (or use your real observations)
true_params = [10.0, 28.0, 8/3]
x0 = [1.0, 1.0, 1.0]
T, dt = 10.0, 0.01
observed_traj = lorenz63_trajectory(*true_params, x0, T, dt)
# Add noise if desired
# observed_traj += np.random.normal(0, 0.1, observed_traj.shape)

# Run MCMC
ndim = 3
nwalkers = 32
initial = np.array([10, 28, 8/3])
p0 = initial + 1e-4 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_probability,
    args=(observed_traj, x0, T, dt)
)

# Burn-in
state = sampler.run_mcmc(p0, 1000, progress=True)
sampler.reset()

# Production run
sampler.run_mcmc(state, 5000, progress=True)

# Extract samples
samples = sampler.get_chain(flat=True)
print(f"Mean: {samples.mean(axis=0)}")
print(f"Std: {samples.std(axis=0)}")

# Visualize
fig = corner.corner(samples, labels=['σ', 'ρ', 'β'], 
                   truths=true_params)